# Project 9 — LLM Inference Server

## Overview
We build a production-style **LLM inference server** with an OpenAI-compatible REST API. The server supports streaming, concurrent requests, and basic request queuing.

**Architecture:**
```
Client → FastAPI Gateway → Request Queue → LLM Backend → Streaming Response
```

## 1. Theory — LLM Serving

### KV Cache
During autoregressive generation, each new token attends to all previous tokens. Without caching, this requires recomputing keys and values for all past tokens at every step. The **KV cache** stores past $K$ and $V$ tensors:
- Without cache: $O(T^2)$ per request
- With cache: $O(T)$ per step after prefill

### Batching Strategies
| Strategy | Description | Pros | Cons |
|---|---|---|---|
| **Static** | Batch N requests, wait for all | Simple | Slow due to padding |
| **Dynamic** | Batch requests within a time window | Better utilisation | More complex |
| **Continuous** | Add/remove requests from batch mid-generation | Best GPU util | Complex implementation |

### Streaming with SSE
**Server-Sent Events (SSE)** allow the server to push data to the client as tokens are generated, instead of waiting for the full response. Format:
```
data: {"choices": [{"delta": {"content": "Hello"}}]}

data: [DONE]
```

### Throughput vs Latency
- **Throughput**: tokens per second (favours large batches)
- **Latency (TTFT)**: time to first token (favours small batches, stream ASAP)
- These objectives conflict — production systems tune based on SLA requirements.

### Quantisation
Reduce memory footprint by representing weights at lower precision:
| Format | Memory vs FP32 | Accuracy loss |
|---|---|---|
| FP16 | 50% | Minimal |
| INT8 | 25% | Small |
| INT4 | 12.5% | Moderate |

In [ ]:
# Required packages:
# pip install fastapi uvicorn httpx

import os
import time
import json
import asyncio
import subprocess
import threading
import numpy as np
import matplotlib.pyplot as plt

print('Base imports done.')

## 2. Server Implementation
We write the server code to a file and run it as a subprocess.

In [ ]:
server_code = '''
"""LLM Inference Server — OpenAI-compatible REST API."""
import asyncio
import time
import json
import random
import logging
from typing import List, Optional, AsyncGenerator
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="MiniLLM Server", version="0.1.0")

# ── Request/Response Models ───────────────────────────────────────────────────

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = "mini-gpt"
    messages: List[Message]
    max_tokens: int = 100
    temperature: float = 1.0
    stream: bool = False

class CompletionRequest(BaseModel):
    model: str = "mini-gpt"
    prompt: str
    max_tokens: int = 100
    temperature: float = 1.0
    stream: bool = False

# ── Simple LLM Mock (replace with real model in production) ──────────────────

RESPONSES = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "Neural networks are inspired by the human brain and consist of interconnected layers of neurons.",
    "Deep learning uses many layers to learn hierarchical representations of data.",
    "The transformer architecture uses self-attention to process sequences in parallel.",
    "Gradient descent optimises model parameters by following the negative gradient of the loss.",
    "Backpropagation computes gradients by applying the chain rule through the computation graph.",
    "Regularisation techniques like dropout and weight decay prevent overfitting.",
    "Transfer learning adapts pre-trained models to new tasks with limited data.",
]

# Metrics storage
REQUEST_TIMES = []
TOTAL_REQUESTS = 0
TOTAL_TOKENS = 0

async def generate_tokens(prompt: str, max_tokens: int, temperature: float) -> AsyncGenerator[str, None]:
    """Simulate token-by-token generation."""
    response_text = random.choice(RESPONSES)
    words = response_text.split()
    n_words = min(len(words), max_tokens // 4 + 5)  # approximate

    for i, word in enumerate(words[:n_words]):
        yield word + (" " if i < n_words - 1 else "")
        await asyncio.sleep(0.02)  # simulate ~50 tokens/sec

# ── Endpoints ────────────────────────────────────────────────────────────────

@app.get("/health")
def health():
    return {"status": "ok", "model": "mini-gpt"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": "mini-gpt", "object": "model"}]}

@app.get("/metrics")
def metrics():
    avg_latency = float(sum(REQUEST_TIMES) / len(REQUEST_TIMES)) if REQUEST_TIMES else 0
    return {
        "total_requests": TOTAL_REQUESTS,
        "total_tokens":   TOTAL_TOKENS,
        "avg_latency_ms": round(avg_latency * 1000, 2),
        "p95_latency_ms": round(float(sorted(REQUEST_TIMES)[int(len(REQUEST_TIMES)*0.95)]) * 1000, 2) if len(REQUEST_TIMES) >= 20 else 0,
    }

@app.post("/v1/chat/completions")
async def chat_completions(request: ChatCompletionRequest):
    global TOTAL_REQUESTS, TOTAL_TOKENS
    t_start = time.time()

    prompt = "\\n".join([f"{m.role}: {m.content}" for m in request.messages])

    if request.stream:
        async def event_stream():
            async for token in generate_tokens(prompt, request.max_tokens, request.temperature):
                chunk = {
                    "object": "chat.completion.chunk",
                    "choices": [{"delta": {"content": token}, "finish_reason": None}]
                }
                yield f"data: {json.dumps(chunk)}\\n\\n"
            yield "data: [DONE]\\n\\n"
        return StreamingResponse(event_stream(), media_type="text/event-stream")

    # Non-streaming
    full_text = ""
    async for token in generate_tokens(prompt, request.max_tokens, request.temperature):
        full_text += token

    latency = time.time() - t_start
    REQUEST_TIMES.append(latency)
    TOTAL_REQUESTS += 1
    TOTAL_TOKENS += len(full_text.split())

    return {
        "object": "chat.completion",
        "choices": [{
            "message": {"role": "assistant", "content": full_text.strip()},
            "finish_reason": "stop"
        }],
        "usage": {"prompt_tokens": len(prompt.split()), "completion_tokens": len(full_text.split())}
    }

@app.post("/v1/completions")
async def completions(request: CompletionRequest):
    global TOTAL_REQUESTS, TOTAL_TOKENS
    t_start = time.time()

    full_text = ""
    async for token in generate_tokens(request.prompt, request.max_tokens, request.temperature):
        full_text += token

    latency = time.time() - t_start
    REQUEST_TIMES.append(latency)
    TOTAL_REQUESTS += 1
    TOTAL_TOKENS += len(full_text.split())

    return {
        "object": "text_completion",
        "choices": [{"text": full_text.strip(), "finish_reason": "stop"}],
        "usage": {"prompt_tokens": len(request.prompt.split()), "completion_tokens": len(full_text.split())}
    }


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080, log_level="warning")
'''

with open('server.py', 'w') as f:
    f.write(server_code)

print('server.py written.')

## 3. Client Implementation

In [ ]:
client_code = '''
"""Simple client to test the LLM inference server."""
import httpx
import json

BASE_URL = "http://localhost:8080"

def health_check():
    r = httpx.get(f"{BASE_URL}/health")
    return r.json()

def chat(messages, stream=False):
    payload = {"model": "mini-gpt", "messages": messages, "max_tokens": 100, "stream": stream}
    if stream:
        with httpx.stream("POST", f"{BASE_URL}/v1/chat/completions", json=payload, timeout=30) as r:
            for line in r.iter_lines():
                if line.startswith("data: ") and line != "data: [DONE]":
                    data = json.loads(line[6:])
                    token = data["choices"][0]["delta"].get("content", "")
                    print(token, end="", flush=True)
        print()
    else:
        r = httpx.post(f"{BASE_URL}/v1/chat/completions", json=payload, timeout=30)
        return r.json()

def complete(prompt):
    payload = {"model": "mini-gpt", "prompt": prompt, "max_tokens": 100}
    r = httpx.post(f"{BASE_URL}/v1/completions", json=payload, timeout=30)
    return r.json()

if __name__ == "__main__":
    print("Health:", health_check())
    print("\nChat (non-streaming):")
    resp = chat([{"role": "user", "content": "What is machine learning?"}])
    print(resp["choices"][0]["message"]["content"])
    print("\nCompletion:")
    resp = complete("Explain neural networks:")
    print(resp["choices"][0]["text"])
'''

with open('client.py', 'w') as f:
    f.write(client_code)

print('client.py written.')

## 4. Start the Server

In [ ]:
import subprocess, time, sys

# Start server as background process
server_proc = subprocess.Popen(
    [sys.executable, 'server.py'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for server to be ready
time.sleep(3)

import httpx
try:
    r = httpx.get('http://localhost:8080/health', timeout=5)
    print('Server status:', r.json())
    print('Server is running!')
except Exception as e:
    print(f'Server not ready: {e}')
    print('Try running: python server.py  in a separate terminal')

## 5. Test the API

In [ ]:
BASE_URL = 'http://localhost:8080'

# Health check
r = httpx.get(f'{BASE_URL}/health')
print('Health:', r.json())

# List models
r = httpx.get(f'{BASE_URL}/v1/models')
print('Models:', r.json())

In [ ]:
# Chat completion — non-streaming
payload = {
    'model': 'mini-gpt',
    'messages': [{'role': 'user', 'content': 'Explain machine learning briefly.'}],
    'max_tokens': 50,
    'stream': False
}
r = httpx.post(f'{BASE_URL}/v1/chat/completions', json=payload, timeout=30)
resp = r.json()
print('Response:', resp['choices'][0]['message']['content'])
print('Usage:', resp['usage'])

In [ ]:
# Streaming response
print('Streaming response:')
print('-' * 40)
payload['stream'] = True

with httpx.stream('POST', f'{BASE_URL}/v1/chat/completions', json=payload, timeout=30) as r:
    for line in r.iter_lines():
        if line.startswith('data: ') and line != 'data: [DONE]':
            data  = json.loads(line[6:])
            token = data['choices'][0]['delta'].get('content', '')
            print(token, end='', flush=True)

print('\n' + '-' * 40)
print('Streaming complete.')

## 6. Concurrency Benchmark

In [ ]:
import concurrent.futures

PROMPTS = [
    'What is deep learning?',
    'Explain gradient descent.',
    'What is a neural network?',
    'How does backpropagation work?',
    'What is the transformer architecture?',
    'Explain attention mechanisms.',
    'What is overfitting?',
    'How does dropout work?',
    'What is transfer learning?',
    'Explain the loss function.',
]

def send_request(prompt: str):
    """Send a single request and return latency."""
    t0 = time.time()
    r  = httpx.post(
        f'{BASE_URL}/v1/completions',
        json={'model': 'mini-gpt', 'prompt': prompt, 'max_tokens': 50},
        timeout=30
    )
    latency = time.time() - t0
    return {'prompt': prompt[:30], 'latency': latency, 'status': r.status_code}


print('Sending 10 concurrent requests...')
t_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(send_request, p) for p in PROMPTS]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

total_time = time.time() - t_start
latencies  = [r['latency'] for r in results]

print(f'\nTotal time for 10 concurrent requests: {total_time:.2f}s')
print(f'Throughput: {len(results)/total_time:.1f} req/s')
print(f'Mean latency: {np.mean(latencies)*1000:.0f} ms')
print(f'P95 latency:  {np.percentile(latencies, 95)*1000:.0f} ms')
print(f'Max latency:  {max(latencies)*1000:.0f} ms')

In [ ]:
# Plot latency histogram
plt.figure(figsize=(8, 4))
plt.hist([l * 1000 for l in latencies], bins=10, color='steelblue', edgecolor='white')
plt.axvline(np.mean(latencies)*1000, color='red', linestyle='--', label=f'Mean: {np.mean(latencies)*1000:.0f}ms')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.title('Request Latency Distribution (10 concurrent requests)')
plt.legend()
plt.tight_layout()
plt.show()

# Server metrics
r = httpx.get(f'{BASE_URL}/metrics')
print('\nServer Metrics:', json.dumps(r.json(), indent=2))

In [ ]:
# Clean up server process
try:
    server_proc.terminate()
    print('Server stopped.')
except:
    pass

## Summary

| Component | Implementation |
|---|---|
| Framework | FastAPI + uvicorn |
| API | OpenAI-compatible `/v1/chat/completions` |
| Streaming | Server-Sent Events (SSE) |
| Concurrency | AsyncIO + thread pool |
| Metrics | `/metrics` endpoint |
| Model | Swap mock with real model |

**To use a real model**: Replace `generate_tokens()` with actual inference using a HuggingFace model or the Anthropic API. The server framework stays unchanged.